# Lab 03 · Função de custo na regressão linear

*Curso 1, semana 1 · Adaptação em português do laboratório “Cost Function”, da Machine Learning Specialization.*

Como escolher a reta que melhor prevê o preço de uma casa? Primeiro vamos calcular o erro de duas previsões à mão. Depois vamos usar o mesmo cálculo para comparar várias retas.

Execute as células em ordem no Jupyter ou no Colab. Os gráficos usam NumPy e Matplotlib. Com `ipywidgets` instalado, você também pode explorar os parâmetros com controles deslizantes.

## 1. Duas casas para começar

Cada posição nos vetores representa a mesma venda. A casa de índice 0 tem mil pés quadrados e foi vendida por 300 mil dólares. A de índice 1 tem dois mil pés quadrados e foi vendida por 500 mil dólares. Mantemos essas unidades em todas as contas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([1.0, 2.0])        # Área em milhares de pés quadrados
y = np.array([300.0, 500.0])    # Preço em milhares de dólares

## 2. Da previsão ao custo

A reta prevê o preço pela expressão $\hat y = wx+b$. A cada mil pés quadrados extras, o preço previsto sobe $w$ milhares de dólares. O parâmetro $b$ é o valor da reta quando $x=0$.

Vamos experimentar $w=150$ e $b=100$. Em cada linha, subtraia o preço observado da previsão.

| Casa | Previsão | Observado | Erro | Erro ao quadrado |
|---|---:|---:|---:|---:|
| 1 | 150 × 1 + 100 = 250 | 300 | −50 | 2.500 |
| 2 | 150 × 2 + 100 = 400 | 500 | −100 | 10.000 |

Elevar ao quadrado impede que erros positivos e negativos se cancelem e dá mais peso aos erros grandes. Para $m$ casas, o laboratório usa metade da média desses quadrados.

$$J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}(wx_i+b-y_i)^2$$

Aqui temos duas casas, então $J=(2.500+10.000)/4=3.125$. Esse número mede o ajuste da reta. O erro individual ainda pode ser lido em milhares de dólares.

In [ ]:
def calcular_custo(x, y, w, b):
    """Calcula metade da média dos erros quadráticos para a reta w*x + b."""
    m = len(x)
    soma = 0.0
    for i in range(m):
        previsao = w * x[i] + b
        erro = previsao - y[i]
        soma += erro ** 2
    return soma / (2 * m)


print(f"Custo com w=150 e b=100 → {calcular_custo(x, y, 150, 100):g}")
print(f"Custo com w=200 e b=100 → {calcular_custo(x, y, 200, 100):g}")

## 3. O que muda quando você altera a inclinação?

Fixe $b=100$. No gráfico da esquerda, compare a reta com as duas vendas. No da direita, cada ponto da curva representa o custo de uma inclinação diferente. A marca laranja mostra o custo da reta escolhida.

Comece em $w=150$ e experimente 200 e 250. Antes de executar, tente prever qual valor faz a reta passar pelas duas casas.

In [ ]:
def explorar_inclinacao(w=150):
    b = 100
    inclinacoes = np.arange(0, 401, 5)
    custos = [calcular_custo(x, y, valor, b) for valor in inclinacoes]
    fig, (dados, curva) = plt.subplots(1, 2, figsize=(10, 4))
    dados.scatter(x, y, label="Vendas")
    dados.plot(x, w * x + b, color="tab:orange", label="Previsão")
    dados.set(xlabel="Área (mil pés²)", ylabel="Preço (mil US$)", ylim=(0, 950))
    dados.legend()
    curva.plot(inclinacoes, custos)
    curva.scatter(w, calcular_custo(x, y, w, b), color="tab:orange", zorder=3)
    curva.set(xlabel="Inclinação w", ylabel="Custo J", title=f"w = {w}, b = {b}")
    fig.tight_layout()
    plt.show()


try:
    from ipywidgets import interact
except ImportError:
    interact = None

if interact is None:
    explorar_inclinacao(w=150)  # Altere w e execute novamente.
else:
    interact(explorar_inclinacao, w=(0, 400, 5))

Com $w=200$ e $b=100$, a reta prevê 300 e 500. Os dois erros valem zero. Como cada parcela da soma é um quadrado, esse é o menor custo possível. Com $w=150$ e $w=250$, os erros têm sinais opostos entre as duas retas, mas os custos são iguais.

## 4. Seis casas e dois parâmetros

Agora os preços observados se espalham em torno de uma tendência. Vamos testar combinações de $w$ e $b$. O código calcula uma grade de custos uma única vez para desenhar o mapa de contorno e a superfície 3D.

In [ ]:
areas = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
precos = np.array([250, 300, 480, 430, 630, 730])

valores_w = np.linspace(80, 320, 81)
valores_b = np.linspace(-200, 200, 81)
grade_w, grade_b = np.meshgrid(valores_w, valores_b)
custos = calcular_custo(areas, precos, grade_w, grade_b)  # Uma conta por par (w, b).

No primeiro gráfico, cada segmento pontilhado liga o preço observado à previsão da reta para aquela casa. O comprimento do segmento é o tamanho do erro. No segundo gráfico, cada curva liga combinações com o mesmo custo. Experimente $w=209$ e $b=2$ e observe a posição da marca em relação ao centro das curvas.

In [ ]:
def explorar_ajuste(w=150, b=100):
    previsoes = w * areas + b
    fig, (dados, mapa) = plt.subplots(1, 2, figsize=(11, 4))
    dados.vlines(areas, precos, previsoes, linestyles="dashed", color="gray")
    dados.scatter(areas, precos, label="Vendas")
    dados.plot(areas, previsoes, color="tab:orange", label="Previsão")
    dados.set(xlabel="Área (mil pés²)", ylabel="Preço (mil US$)")
    dados.legend()
    niveis = [1600, 2000, 3000, 5000, 10000, 25000, 50000]
    linhas = mapa.contour(grade_w, grade_b, custos, levels=niveis)
    mapa.clabel(linhas, fontsize=8)
    mapa.scatter(w, b, color="tab:orange")
    mapa.set(xlabel="Inclinação w", ylabel="Intercepto b",
             title=f"Custo da reta = {calcular_custo(areas, precos, w, b):.1f}")
    fig.tight_layout()
    plt.show()


if interact is None:
    explorar_ajuste(w=150, b=100)  # Altere os dois valores para comparar retas.
else:
    interact(explorar_ajuste, w=(80, 320), b=(-200, 200))

### O mesmo mapa visto de lado

Imagine levantar cada ponto do mapa até a altura correspondente ao seu custo. Surge uma superfície em forma de tigela. O fundo indica os parâmetros de melhor ajuste. Com estas seis casas, até a melhor reta deixa alguma distância entre as previsões e os preços observados.

In [ ]:
fig = plt.figure(figsize=(8, 5))
eixo = fig.add_subplot(projection="3d")
eixo.plot_surface(grade_w, grade_b, custos, cmap="viridis")
eixo.set(xlabel="Inclinação w", ylabel="Intercepto b", zlabel="Custo J")
plt.show()

## Para revisar sem olhar o código

1. Por que a divisão por $2m$ acontece depois do laço?
2. Nas duas primeiras casas, o que acontece com as previsões se você mantiver $w=200$ e aumentar $b$ de 100 para 110? Calcule o custo antes de conferir com `calcular_custo(x, y, 200, 110)`.
3. Nas seis casas, o que precisaria acontecer com os segmentos pontilhados para que o custo fosse zero?

**Confira suas respostas.** Primeiro somamos a contribuição de todas as casas e depois dividimos pelo mesmo fator. No segundo exercício, as duas previsões sobem 10, então o custo é $(10^2+10^2)/4=50$. No terceiro, todos os segmentos precisariam ter comprimento zero ao mesmo tempo.